# EdNet 전처리

`02_build_tables`에서 만든 테이블에 연결만 해서 분석합니다. 재시작해도 테이블을 다시 만들지 않습니다.

## 0. DB 연결
`read_only=True`: 분석 중 실수로 테이블을 덮어쓰지 않도록 읽기 전용으로 엽니다.

In [2]:
import duckdb

con = duckdb.connect("../data/ednet.duckdb", read_only=True)

### 데이터 미리보기
KT4는 유저의 행동(문제 진입, 답 선택, 제출 등)이 한 행씩 쌓인 테이블이고, 컬럼은 8개입니다.

In [3]:
con.sql("SELECT * FROM kt4 LIMIT 10").df()

,user_id,timestamp,action_type,item_id,cursor_time,source,user_answer,platform
0,u1,1565096151269,enter,b3544,<NA>,diagnosis,NaN,mobile
1,u1,1565096187972,respond,q5012,<NA>,diagnosis,b,mobile
2,u1,1565096194904,submit,b3544,<NA>,diagnosis,NaN,mobile
3,u1,1565096195001,enter,b3238,<NA>,diagnosis,NaN,mobile
4,u1,1565096218682,respond,q4706,<NA>,diagnosis,c,mobile
5,u1,1565096223029,submit,b3238,<NA>,diagnosis,NaN,mobile
6,u1,1565096223119,enter,b2898,<NA>,diagnosis,NaN,mobile
7,u1,1565096290094,respond,q4366,<NA>,diagnosis,b,mobile
8,u1,1565096295925,submit,b2898,<NA>,diagnosis,NaN,mobile
9,u1,1565096296012,enter,b3361,<NA>,diagnosis,NaN,mobile


## 1. `user_id` 검증

### 1-1. 결측치·형식 확인
빈 값이 없는지, 모두 `u숫자` 형식인지 확인합니다.
유저 수가 공식 문서의 KT4 유저 수(297,915명)와 같습니다.

| 쿼리 행 | 설명 |
|---|---|
| `COUNT(*) FILTER (WHERE user_id IS NULL)` | 값이 비어 있는 행 수 |
| `COUNT(*) FILTER (WHERE NOT regexp_matches(user_id, '^u[0-9]+$'))` | `u` + 숫자 형식이 아닌 행 수 |
| `COUNT(DISTINCT user_id)` | 서로 다른 유저 수 |
| `FROM kt4` | 전체 KT4 테이블 대상 |

In [4]:
con.sql("""
SELECT
    COUNT(*) FILTER (WHERE user_id IS NULL) AS null_count,
    COUNT(*) FILTER (WHERE NOT regexp_matches(user_id, '^u[0-9]+$')) AS bad_format_count,
    COUNT(DISTINCT user_id) AS user_count
FROM kt4
""").df()

,null_count,bad_format_count,user_count
0,0,0,297915


## 2. `timestamp` 검증

### 2-1. 유닉스 밀리초 형식 확인

| 쿼리 행 | 설명 |
|---|---|
| `COUNT(*) FILTER (WHERE timestamp IS NULL)` | 값이 비어 있는 행 수 |
| `COUNT(*) FILTER (WHERE timestamp < 1e12 OR timestamp >= 1e13)` | 13자리가 아닌 행 수 = 밀리초가 아닌 값 (초 단위면 10자리, 마이크로초면 16자리) |
| `epoch_ms(MIN(timestamp))` | 가장 이른 값을 밀리초로 보고 날짜로 변환 → 날짜가 상식적이면 밀리초가 맞다는 근거 |
| `epoch_ms(MAX(timestamp))` | 가장 늦은 값을 날짜로 변환 |
| `FROM kt4` | 전체 KT4 테이블 대상 |

In [5]:
con.sql("""
SELECT
    COUNT(*) FILTER (WHERE timestamp IS NULL) AS null_count,
    COUNT(*) FILTER (WHERE timestamp < 1e12 OR timestamp >= 1e13) AS not_ms_count,
    epoch_ms(MIN(timestamp)) AS min_time,
    epoch_ms(MAX(timestamp)) AS max_time
FROM kt4
""").df()

,null_count,not_ms_count,min_time,max_time
0,0,0,2018-08-27 03:00:44.464,2019-12-01 21:15:34.502
